### Per the discussion at the end of the <i><span style="color:red">previous</span></i> <span style="color:blue">Gemini</span> notebook:
#### if you run the two identical prompts against each other, <span style="color:orange">ChatGPT</span><span style="color:green"> wins</span>... by a <i>longshot</i>.

#### Which makes sense because, well, that prompt was engineered for <span style="color:orange">OpenAI</span>'s 4o-mini engine.

#### What follows is the redo of the code to see if <span style="color:blue">Gemini</span> can use all that it learned from  <span style="color:red">YouTube</span> videos and <span style="color:blue">Google Docs</span> to do a better job at resume optimization than <span style="color:orange">OpenAI</span> /  <span style="color:orange">OpenAI</span>ChatGPT</span>.
***
### Step 1: Imports
***

In [8]:
# working with our os and .env file
import os
from dotenv import load_dotenv

# since we're using Gemini
import google.generativeai as genai

# make printouts look nicer
from IPython.display import display, Markdown
from markdown import markdown

# HTML this to a PDF
from weasyprint import HTML


In [9]:
# load .env stuff - looking for 'True'
load_dotenv()

True

### Step 2: Load the Resume
***

In [10]:
with open("/Users/nicholasjoseph/Desktop/nj_gemini_pm_res.md", "r", encoding="utf-8") as resume_file:
    resume_string = resume_file.read()

### Step 3: Input the Job Description
***

In [4]:
jd_string = input("Paste the job description here:\n")

Paste the job description here:
 About the job Overview  Leverage your consulting and project management expertise to win opportunities and deliver high-quality technical consulting engagements for Esri customers. In this role, you will lead teams through all business development and implementation phases, including requirements gathering, analysis, design, build/configuration, and deployment. By collaborating closely with industry experts across the company, you will play a pivotal role in enabling customer success.  As a Project Manager, You Will Be Responsible For  Engaging with key stakeholders on opportunity development, proposal development, and project execution while managing expectations  Implementing project scope through the coordination of professional services subject matter experts, and our customers  Managing project teams comprised of consultants and technical staff dedicated to supporting customers and their mission-critical systems and applications   Responsibilities 

### <u>Step 4 - The Meat: Define the Function Calling Schema</u>
#### The prompt in the original <span style="color:orange">ChatGPT</span> notebook was great for giving information, but it just generated a giant blob of text.
#### <span style="color:blue">Gemini</span> works a little differently. Instead of just asking for a big block of text, we can send a structured request.
#### The goal is not only to optimize the content of the resume, but also to get suggestions.
#### Structured requests explicitly state we want these things back; in the original prompt, we were only <i>hoping</i> to get those things back.
***

In [11]:
# demand the Python dictionary be the object, with "tailored_resume" and "additional_suggestions"
schema = {
    "type": "object",
    "properties": {
        "tailored_resume": {
            "type": "string",
            "description": "The resume customized to the job description, formatted in Markdown."
        },
        "additional_suggestions": {
            "type": "string",
            "description": "Things that may improve your resume (skills, certs, etc.): "
        }
    },
    "required": ["tailored_resume"]
}

### Step 5: Write the Prompt
***

In [12]:
# again: functioning on the fly
prompt_template = lambda resume_string, job_desc_string: f"""
You are a professional resume optimization expert. 
Optimize the resume provided to align with the given job description following the guidelines below.

1. Make the resume one page, relevant, keyword optimized, action-driven.
2. Format it cleanly in Markdown.
3. At the end, suggest improvements if gaps exist.

Resume:
{resume_string}

Job Description:
{jd_string}
"""

prompt = prompt_template(resume_string, jd_string)


### Step 6: Put <span style="color:blue">Gemini</span> To Work:
#### <u> Really, a two-part Step</u>:
> #### a.) Configure the <span style="color:blue">Gemini</span> client; and
> #### b.) Choose the <span style="color:blue">Gemini</span> model to use
***

In [13]:
# Step 6a:
gemini_api_key = os.environ.get("GEMINI_API_KEY")

# set up a reminder to run 'load_dotenv()
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY environment variable is not set. Please set it and restart the kernel.")

genai.configure(api_key=gemini_api_key)

In [15]:
# step 6b:

# choose the engine:
model = genai.GenerativeModel("gemini-2.0-flash")

# the actual function call
response = model.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(temperature=0.7),
    tools=[{
        "function_declarations": [
            {
                "name": "optimize_resume",
                "description": "Tailor a resume to a target job description.",
                "parameters": schema
            }
        ]
    }]
)

### Step 7: Get the Results
***

In [16]:
# function_args = response.candidates[0].content.parts[0].function_call.args

# resume_text = function_args['tailored_resume']
# suggestions_text = function_args.get('additional_suggestions', "No additional suggestions were provided.")

TypeError: 'NoneType' object is not subscriptable

#### ^^ didn't work! Looks like the model didn't respond with a function call - leads to a 'NoneType' object. 
#### <u>So... what?</u>
##### 1. Set up a function schema that can be passed into the model; and
##### 2. Then force the model to return with a function call.

In [ ]:
import os
import google.generativeai as genai
from IPython.display import display, Markdown
from markdown import markdown
from weasyprint import HTML

# Step 1: Load resume and job description
with open("/Users/nicholasjoseph/Desktop/nj_gemini_pm_res.md", "r", encoding="utf-8") as resume_file:
    resume_string = resume_file.read()

job_desc_string = input("Paste the job description:\n")

# Step 2: Define the function calling schema
schema = {
    "type": "object",
    "properties": {
        "tailored_resume": {
            "type": "string",
            "description": "The resume customized to the job description, formatted in Markdown."
        },
        "additional_suggestions": {
            "type": "string",
            "description": "Additional improvements like skills, certifications, or projects."
        }
    },
    "required": ["tailored_resume"]
}

# Step 3: Write the system prompt
prompt_template = lambda resume_string, job_desc_string: f"""
You are a professional resume optimization expert. 
Optimize the resume provided to align with the given job description following the guidelines below.

1. Make the resume one page, relevant, keyword optimized, action-driven.
2. Format it cleanly in Markdown.
3. At the end, suggest improvements if gaps exist.

Resume:
{resume_string}

Job Description:
{job_desc_string}
"""

prompt = prompt_template(resume_string, job_desc_string)

# Step 4: Configure the Gemini client
gemini_api_key = os.environ.get("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY environment variable is not set. Please set it and restart the kernel.")

genai.configure(api_key=gemini_api_key)

# Step 5: Instantiate the model and call with function declaration
model = genai.GenerativeModel("gemini-2.0-pro")

response = model.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(temperature=0.7),
    tools=[{
        "function_declarations": [
            {
                "name": "optimize_resume",
                "description": "Tailor a resume to a target job description.",
                "parameters": schema
            }
        ]
    }]
)

# Step 6: Extract the results from the structured response
function_args = response.candidates[0].content.parts[0].function_call.args

resume_text = function_args['tailored_resume']
suggestions_text = function_args.get('additional_suggestions', "No additional suggestions were provided.")

# Step 7: Display the results nicely
display(Markdown(resume_text))
display(Markdown("## Additional Suggestions\n" + suggestions_text))

# Step 8: Save to Markdown and PDF
output_dir = "google_resumes"
os.makedirs(output_dir, exist_ok=True)

# Save markdown
markdown_output = os.path.join(output_dir, "nj_pm_resume.md")
with open(markdown_output, "w", encoding="utf-8") as file:
    file.write(resume_text)

# Save PDF
html_content = markdown(resume_text)
output_pdf_file = os.path.join(output_dir, "nj_pm_resume.pdf")
HTML(string=html_content).write_pdf(output_pdf_file)
